In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_customers_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_sellers_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_order_reviews_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_order_items_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_products_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_geolocation_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/product_category_name_translation.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_orders_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_order_payments_dataset.csv


In [2]:
import pandas as pd

path = "/kaggle/input/datasets/olistbr/brazilian-ecommerce/"

orders = pd.read_csv(path + "olist_orders_dataset.csv")
customers = pd.read_csv(path + "olist_customers_dataset.csv")
items = pd.read_csv(path + "olist_order_items_dataset.csv")
payments = pd.read_csv(path + "olist_order_payments_dataset.csv")
reviews = pd.read_csv(path + "olist_order_reviews_dataset.csv")
products = pd.read_csv(path + "olist_products_dataset.csv")
sellers = pd.read_csv(path + "olist_sellers_dataset.csv")

tables = {
    "orders": orders,
    "customers": customers,
    "items": items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
}

for name, df in tables.items():
    print(name, df.shape)

orders (99441, 8)
customers (99441, 5)
items (112650, 7)
payments (103886, 5)
reviews (99224, 7)
products (32951, 9)
sellers (3095, 4)


In [3]:
for name, df in tables.items():
    print(name, "| missing:", df.isna().sum().sum(), "| duplicate rows:", df.duplicated().sum())

orders | missing: 4908 | duplicate rows: 0
customers | missing: 0 | duplicate rows: 0
items | missing: 0 | duplicate rows: 0
payments | missing: 0 | duplicate rows: 0
reviews | missing: 145903 | duplicate rows: 0
products | missing: 2448 | duplicate rows: 0
sellers | missing: 0 | duplicate rows: 0


In [4]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c])

print(orders.dtypes)

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [5]:
delivered = orders[orders["order_status"] == "delivered"].dropna(
    subset=["order_delivered_customer_date"]
).copy()

delivered["delivery_days"] = (
    delivered["order_delivered_customer_date"] - delivered["order_purchase_timestamp"]
).dt.days

delivered["delivered_on_time"] = (
    delivered["order_delivered_customer_date"] <= delivered["order_estimated_delivery_date"]
)

print(len(orders), "->", len(delivered))
print("On time %:", round(delivered["delivered_on_time"].mean() * 100, 1))

99441 -> 96470
On time %: 91.9


In [6]:
pay = payments.groupby("order_id", as_index=False)["payment_value"].sum()
rev = reviews.groupby("order_id", as_index=False)["review_score"].mean()
itm = items.groupby("order_id", as_index=False).agg(
    items_count=("order_item_id", "count"),
    price=("price", "sum"),
    freight=("freight_value", "sum"),
)

master = (
    delivered.merge(customers, on="customer_id", how="left")
             .merge(itm, on="order_id", how="left")
             .merge(pay, on="order_id", how="left")
             .merge(rev, on="order_id", how="left")
)

print(master.shape)
print(master.isna().sum())

(96470, 19)
order_id                           0
customer_id                        0
order_status                       0
order_purchase_timestamp           0
order_approved_at                 14
order_delivered_carrier_date       1
order_delivered_customer_date      0
order_estimated_delivery_date      0
delivery_days                      0
delivered_on_time                  0
customer_unique_id                 0
customer_zip_code_prefix           0
customer_city                      0
customer_state                     0
items_count                        0
price                              0
freight                            0
payment_value                      1
review_score                     646
dtype: int64


In [7]:
import sqlite3

conn = sqlite3.connect("olist.db")

to_load = {
    "orders": orders,
    "customers": customers,
    "order_items": items,
    "order_payments": payments,
    "order_reviews": reviews,
    "products": products,
    "sellers": sellers,
    "orders_master": master,
}

for name, df in to_load.items():
    df.to_sql(name, conn, if_exists="replace", index=False)

print("loaded", len(to_load), "tables")

loaded 8 tables


In [8]:
for name in to_load:
    n = pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {name}", conn)["n"][0]
    print(name, n)

orders 99441
customers 99441
order_items 112650
order_payments 103886
order_reviews 99224
products 32951
sellers 3095
orders_master 96470


In [9]:
def q(sql):
    return pd.read_sql_query(sql, conn)

In [10]:
q("""
SELECT customer_state,
       COUNT(*) AS orders,
       ROUND(AVG(delivery_days), 1) AS avg_days
FROM orders_master
GROUP BY customer_state
ORDER BY avg_days DESC
LIMIT 10
""")

,customer_state,orders,avg_days
0,RR,41,29.0
1,AP,67,26.7
2,AM,145,26.0
3,AL,397,24.0
4,PA,946,23.3
5,MA,717,21.1
6,SE,335,21.0
7,CE,1279,20.8
8,AC,80,20.6
9,PB,517,20.0


In [11]:
q("""
SELECT customer_state,
       COUNT(*) AS orders,
       ROUND(AVG(delivery_days), 1) AS avg_days
FROM orders_master
WHERE customer_state = 'SP'
""")

,customer_state,orders,avg_days
0,SP,40494,8.3


In [12]:
q("""
SELECT customer_state,
       COUNT(*) AS orders,
       ROUND(100.0 * AVG(delivered_on_time), 1) AS on_time_pct
FROM orders_master
GROUP BY customer_state
ORDER BY on_time_pct ASC
LIMIT 10
""")

,customer_state,orders,on_time_pct
0,AL,397,76.1
1,MA,717,80.3
2,PI,476,84.0
3,CE,1279,84.7
4,SE,335,84.8
5,BA,3256,86.0
6,RJ,12350,86.5
7,TO,274,87.2
8,PA,946,87.6
9,ES,1995,87.8


In [13]:
q("""
SELECT strftime('%Y-%m', order_purchase_timestamp) AS month,
       COUNT(*) AS orders,
       ROUND(SUM(payment_value), 0) AS revenue
FROM orders_master
GROUP BY month
ORDER BY month
""")

,month,orders,revenue
0,2016-09,1,NaN
1,2016-10,265,46567.0
2,2016-12,1,20.0
3,2017-01,750,127546.0
4,2017-02,1653,271299.0
5,2017-03,2546,414369.0
6,2017-04,2303,390952.0
7,2017-05,3545,566873.0
8,2017-06,3135,490226.0
9,2017-07,3872,566404.0


In [14]:
q("""
SELECT p.product_category_name,
       COUNT(*) AS items_sold,
       ROUND(SUM(oi.price), 0) AS revenue
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
JOIN orders_master m ON oi.order_id = m.order_id
GROUP BY p.product_category_name
ORDER BY revenue DESC
LIMIT 10
""")

,product_category_name,items_sold,revenue
0,beleza_saude,9465,1233132.0
1,relogios_presentes,5857,1165899.0
2,cama_mesa_banho,10953,1023435.0
3,esporte_lazer,8430,954674.0
4,informatica_acessorios,7643,888614.0
5,moveis_decoracao,8160,711928.0
6,utilidades_domesticas,6795,615629.0
7,cool_stuff,3718,610204.0
8,automotivo,4139,578849.0
9,brinquedos,4029,471097.0


In [15]:
q("""
SELECT customer_state,
       COUNT(*) AS orders,
       ROUND(AVG(julianday(order_estimated_delivery_date) - julianday(order_purchase_timestamp)), 1) AS avg_promised_days,
       ROUND(AVG(delivery_days), 1) AS avg_actual_days
FROM orders_master
WHERE customer_state IN ('RR','AP','AM','AL','MA','SP','RJ')
GROUP BY customer_state
ORDER BY avg_actual_days DESC
""")

,customer_state,orders,avg_promised_days,avg_actual_days
0,RR,41,46.0,29.0
1,AP,67,46.2,26.7
2,AM,145,45.3,26.0
3,AL,397,32.6,24.0
4,MA,717,30.5,21.1
5,RJ,12350,26.4,14.8
6,SP,40494,19.1,8.3


In [16]:
q("""
SELECT delivered_on_time,
       COUNT(*) AS orders,
       ROUND(AVG(review_score), 2) AS avg_review
FROM orders_master
GROUP BY delivered_on_time
""")

,delivered_on_time,orders,avg_review
0,0,7826,2.57
1,1,88644,4.29


In [17]:
q("""
SELECT strftime('%Y-%m', order_purchase_timestamp) AS month,
       COUNT(*) AS orders,
       ROUND(SUM(payment_value), 0) AS revenue
FROM orders_master
GROUP BY month
ORDER BY month
""")

,month,orders,revenue
0,2016-09,1,NaN
1,2016-10,265,46567.0
2,2016-12,1,20.0
3,2017-01,750,127546.0
4,2017-02,1653,271299.0
5,2017-03,2546,414369.0
6,2017-04,2303,390952.0
7,2017-05,3545,566873.0
8,2017-06,3135,490226.0
9,2017-07,3872,566404.0


In [18]:
q("""
SELECT COUNT(*) AS orders_without_payment
FROM orders_master
WHERE payment_value IS NULL
""")

,orders_without_payment
0,1


In [19]:
q("""
SELECT order_id, order_status, order_purchase_timestamp, price, payment_value
FROM orders_master
WHERE payment_value IS NULL
""")

,order_id,order_status,order_purchase_timestamp,price,payment_value
0,bfbd0f9bdef84302105ad712db648a6c,delivered,2016-09-15 12:16:38,134.97,None


In [20]:
q("""
SELECT strftime('%Y-%m', order_purchase_timestamp) AS month,
       COUNT(*) AS orders,
       ROUND(SUM(COALESCE(payment_value, 0)), 0) AS revenue
FROM orders_master
GROUP BY month
ORDER BY month
""")

,month,orders,revenue
0,2016-09,1,0.0
1,2016-10,265,46567.0
2,2016-12,1,20.0
3,2017-01,750,127546.0
4,2017-02,1653,271299.0
5,2017-03,2546,414369.0
6,2017-04,2303,390952.0
7,2017-05,3545,566873.0
8,2017-06,3135,490226.0
9,2017-07,3872,566404.0
